# TorchVision 全组件系统讲解（完整可执行版）

#### 教程概述

本教程系统全面地讲解 TorchVision 的 **6 大核心组件** + 若干辅助子模块：

| 组件 | 功能 | 章节 |
|------|------|------|
| `torchvision.datasets` | 数据集加载与组织 | 第一章 |
| `torchvision.transforms` | 图像预处理与增强 | 第二章 |
| `torchvision.models` | 预训练模型库 | 第三章 |
| `torchvision.utils` | 可视化工具 | 第四章 |
| `torchvision.ops` | 底层视觉算子 | 第五章 |
| `torchvision.io` | 张量式 IO 读写 | 第六章 |

**所有代码均可直接执行**，数据集会自动下载到 `./data/` 目录。

In [1]:
# ============================================================
# 环境准备与版本检查
# ============================================================

import torch
import torchvision
import numpy as np
from PIL import Image
import os
import tempfile
import time

print("=" * 70)
print("环境信息")
print("=" * 70)
print(f"PyTorch 版本: {torch.__version__}")
print(f"TorchVision 版本: {torchvision.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 设备: {torch.cuda.get_device_name(0)}")

# 确保数据目录存在
os.makedirs('./data', exist_ok=True)
print(f"\n数据目录: ./data/")

环境信息
PyTorch 版本: 2.7.1+cu118
TorchVision 版本: 0.22.1+cu118
CUDA 可用: True
CUDA 设备: NVIDIA GeForce GTX 1060 3GB

数据目录: ./data/


#### 第一章：torchvision.datasets — 数据集加载

#### 1.1 内置标准数据集加载

In [2]:
# ============================================================
# 1.1 基础数据集加载示例
# ============================================================

from torchvision import datasets

print("=" * 60)
print("1.1 基础数据集加载示例")
print("=" * 60)

# ---------- MNIST 手写数字 ----------
print("\n>>> MNIST 示例")
mnist_train = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=None
)
mnist_test = datasets.MNIST(root='./data', train=False, download=True)
print(f"  训练集大小: {len(mnist_train)}")
print(f"  测试集大小: {len(mnist_test)}")
print(f"  样本形状: {mnist_train[0][0].size} (PIL Image)")
print(f"  标签: {mnist_train[0][1]}")

# ---------- CIFAR10 ----------
print("\n>>> CIFAR10 示例")
cifar10_train = datasets.CIFAR10(root='./data', train=True, download=True)
cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True)
print(f"  训练集大小: {len(cifar10_train)}")
print(f"  类别: {cifar10_train.classes}")
print(f"  类别索引映射: {cifar10_train.class_to_idx}")
print(f"  样本形状: {cifar10_train[0][0].size}")

# ---------- FashionMNIST ----------
print("\n>>> FashionMNIST 示例")
fashion_train = datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"  类别: {fashion_train.classes}")
print(f"  类别数: {len(fashion_train.classes)}")

# ---------- SVHN ----------
print("\n>>> SVHN 示例")
svhn_train = datasets.SVHN(root='./data', split='train', download=True)
svhn_test = datasets.SVHN(root='./data', split='test', download=True)
print(f"  训练集大小: {len(svhn_train)}")
print(f"  测试集大小: {len(svhn_test)}")
print(f"  标签: {svhn_train[0][1]}")

1.1 基础数据集加载示例

>>> MNIST 示例
  训练集大小: 60000
  测试集大小: 10000
  样本形状: (28, 28) (PIL Image)
  标签: 5

>>> CIFAR10 示例
  训练集大小: 50000
  类别: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  类别索引映射: {'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}
  样本形状: (32, 32)

>>> FashionMNIST 示例
  类别: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
  类别数: 10

>>> SVHN 示例
  训练集大小: 73257
  测试集大小: 26032
  标签: 1


#### 1.2 自定义数据集：ImageFolder

In [3]:
# ============================================================
# 1.2 ImageFolder：按文件夹组织自定义分类数据
# ============================================================

from torchvision.datasets import ImageFolder

print("\n" + "=" * 60)
print("1.2 ImageFolder 自定义数据集")
print("=" * 60)

# 使用临时目录演示
with tempfile.TemporaryDirectory() as tmpdir:
    # 创建目录结构：root/cat/xxx.jpg, root/dog/xxx.jpg
    classes = ['cat', 'dog', 'bird']
    for cls in classes:
        os.makedirs(os.path.join(tmpdir, cls), exist_ok=True)
        for i in range(3):
            img = Image.fromarray(
                np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)
            )
            img.save(os.path.join(tmpdir, cls, f'{i}.jpg'))
    
    # 加载数据集
    dataset = ImageFolder(root=tmpdir)
    
    print(f"\n📁 目录结构：")
    for cls in classes:
        cnt = len(os.listdir(os.path.join(tmpdir, cls)))
        print(f"   ├── {cls}/  ({cnt} 张图片)")
    
    print(f"\n📊 加载结果：")
    print(f"   总样本数: {len(dataset)}")
    print(f"   类别列表: {dataset.classes}")
    print(f"   类别→索引: {dataset.class_to_idx}")
    
    print(f"\n⚠️  标签由文件夹名【字母顺序】决定！")
    print(f"   排序后: {sorted(classes)}")
    print(f"   所以: bird=0, cat=1, dog=2")


1.2 ImageFolder 自定义数据集

📁 目录结构：
   ├── cat/  (3 张图片)
   ├── dog/  (3 张图片)
   ├── bird/  (3 张图片)

📊 加载结果：
   总样本数: 9
   类别列表: ['bird', 'cat', 'dog']
   类别→索引: {'bird': 0, 'cat': 1, 'dog': 2}

⚠️  标签由文件夹名【字母顺序】决定！
   排序后: ['bird', 'cat', 'dog']
   所以: bird=0, cat=1, dog=2


#### 1.3 数据集划分：random_split 和 Subset

In [4]:
# ============================================================
# 1.3 数据集划分：random_split 和 Subset
# ============================================================

from torch.utils.data import random_split, Subset

print("\n" + "=" * 60)
print("1.3 数据集划分")
print("=" * 60)

# 使用 CIFAR10 训练集进行划分
full_dataset = datasets.CIFAR10(root='./data', train=True, download=True)
print(f"原始数据集大小: {len(full_dataset)}")

# ---------- random_split：随机划分 ----------
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_set, val_set = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # 固定种子保证可复现
)

print(f"\nrandom_split 划分（8:2）：")
print(f"  训练集: {len(train_set)}")
print(f"  验证集: {len(val_set)}")

# ---------- Subset：按索引截取 ----------
indices = list(range(100))  # 取前100个
subset = Subset(full_dataset, indices)
print(f"\nSubset 截取（前100个）：")
print(f"  子集大小: {len(subset)}")


1.3 数据集划分
原始数据集大小: 50000

random_split 划分（8:2）：
  训练集: 40000
  验证集: 10000

Subset 截取（前100个）：
  子集大小: 100


#### 1.4 检测与分割数据集简介

In [5]:
# ============================================================
# 1.4 检测与分割数据集
# ============================================================

print("\n" + "=" * 60)
print("1.4 检测与分割数据集")
print("=" * 60)

print("""
📦 Pascal VOC 检测数据集：
   datasets.VOCDetection(
       root='./data',
       year='2012',
       image_set='train',
       download=True
   )
   返回: (PIL Image, target_dict)  # target_dict 含 bbox、label

📦 COCO 检测数据集（需手动下载）：
   datasets.CocoDetection(
       root='./coco/train2017',
       annFile='./coco/annotations/instances_train2017.json'
   )

📦 Cityscapes 语义分割（需注册下载）：
   datasets.Cityscapes(
       root='./cityscapes',
       split='train',
       mode='fine',
       target_type='semantic'
   )
""")


1.4 检测与分割数据集

📦 Pascal VOC 检测数据集：
   datasets.VOCDetection(
       root='./data',
       year='2012',
       image_set='train',
       download=True
   )
   返回: (PIL Image, target_dict)  # target_dict 含 bbox、label

📦 COCO 检测数据集（需手动下载）：
   datasets.CocoDetection(
       root='./coco/train2017',
       annFile='./coco/annotations/instances_train2017.json'
   )

📦 Cityscapes 语义分割（需注册下载）：
   datasets.Cityscapes(
       root='./cityscapes',
       split='train',
       mode='fine',
       target_type='semantic'
   )



#### 第二章：transforms 图像预处理与增强

#### 2.1 transforms.v1 基础操作

In [ ]:
# ============================================================
# 2.1 transforms.v1 基础操作
# ============================================================

from torchvision import transforms
from torchvision.transforms import functional as F

print("=" * 60)
print("2.1 transforms.v1 基础操作")
print("=" * 60)

# 创建测试图片
demo_img = Image.fromarray(
    np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
)
print(f"原始图片: {demo_img.size}")

# ---------- 标准分类训练流水线 ----------
print("\n>>> 分类训练流水线")
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33))
])

result = train_transform(demo_img)
print(f"  输出形状: {result.shape}")
print(f"  值范围: [{result.min():.3f}, {result.max():.3f}]")

# ---------- 验证/测试流水线 ----------
print("\n>>> 验证/测试流水线")
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print(f"  Resize(256) → CenterCrop(224) 是标准做法")

# ---------- 自动增强策略 ----------
print("\n>>> 自动增强策略")
auto_aug = transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.IMAGENET)
rand_aug = transforms.RandAugment(num_ops=2, magnitude=9)
trivial_aug = transforms.TrivialAugmentWide()  # 最新推荐，无参数
print(f"  AutoAugment: {type(auto_aug).__name__}")
print(f"  RandAugment: num_ops=2, magnitude=9")
print(f"  TrivialAugmentWide: 无参数，性能优于 RandAugment")

#### 2.2 函数式 API（functional）

In [ ]:
# ============================================================
# 2.2 函数式 API (functional)
# ============================================================

from torchvision.transforms import functional as F

print("\n" + "=" * 60)
print("2.2 函数式 API")
print("=" * 60)

# 先将图片转为 Tensor
img_tensor = transforms.ToTensor()(demo_img)
print(f"Tensor 形状: {img_tensor.shape}")

# 各种单次变换
print("\n>>> 单次变换示例：")

# 水平翻转
flipped = F.hflip(img_tensor)
print(f"  水平翻转: {flipped.shape}")

# 调整亮度
brightened = F.adjust_brightness(img_tensor, brightness_factor=1.5)
print(f"  调整亮度 (1.5x): {brightened.shape}")

# 高斯模糊
blurred = F.gaussian_blur(img_tensor, kernel_size=5)
print(f"  高斯模糊 (5x5): {blurred.shape}")

# 旋转
rotated = F.rotate(img_tensor, angle=45)
print(f"  旋转 45°: {rotated.shape}")

# 标准化
normalized = F.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
print(f"  标准化: {normalized.shape}")

#### 2.3 transforms.v2 同步变换（检测/分割任务）

In [ ]:
# ============================================================
# 2.3 transforms.v2 同步变换
# v2 支持图像 + bbox + mask + label 同步变换
# ============================================================

from torchvision.transforms import v2
from torchvision import tv_tensors

print("\n" + "=" * 60)
print("2.3 transforms.v2 同步变换")
print("=" * 60)

# 创建模拟检测数据
img = tv_tensors.Image(torch.randint(0, 256, (3, 480, 640), dtype=torch.uint8))

boxes = tv_tensors.BoundingBoxes(
    torch.tensor([[50, 50, 200, 300], [300, 200, 500, 400]], dtype=torch.float32),
    format="XYXY",
    canvas_size=(480, 640)
)

masks = tv_tensors.Mask(torch.zeros(2, 480, 640, dtype=torch.uint8))
masks[0, 50:200, 50:250] = 1
masks[1, 200:400, 300:500] = 1

labels = torch.tensor([1, 2])

print("原始数据：")
print(f"  Image: {img.shape}")
print(f"  Boxes: {boxes.shape}, format={boxes.format}")
print(f"  Masks: {masks.shape}")
print(f"  Labels: {labels}")

# v2 同步变换流水线
sync_transform = v2.Compose([
    v2.RandomHorizontalFlip(p=1.0),
    v2.RandomResizedCrop((300, 400), scale=(0.8, 1.0)),
    v2.SanitizeBoundingBoxes(),
    v2.ToDtype(torch.float32, scale=True)
])

print("\n应用同步变换后：")
transformed = sync_transform(img, {"boxes": boxes, "masks": masks, "labels": labels})

new_img = transformed[0]
new_targets = transformed[1]

print(f"  Image: {new_img.shape}")
print(f"  Boxes: {new_targets['boxes'].shape}, format={new_targets['boxes'].format}")
print(f"  Masks: {new_targets['masks'].shape}")
print(f"  Labels: {new_targets['labels']}")

print("\n✨ v2 核心优势：所有变换自动同步，无需手动处理！")

#### 第三章：torchvision.models — 预训练模型库

#### 3.1 图像分类模型加载

In [6]:
# ============================================================
# 3.1 图像分类模型加载
# ============================================================

from torchvision import models
from torchvision.models import (
    resnet50, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
    vit_b_16, ViT_B_16_Weights,
    swin_t, Swin_T_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights
)

print("=" * 60)
print("3.1 图像分类模型加载")
print("=" * 60)

# 使用新式权重 API（推荐）
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

print(f"\n>>> ResNet50")
print(f"  参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"  配套预处理: {weights.transforms()}")
print(f"  权重元数据: {weights.meta}")

print(f"\n>>> 可用权重选项:")
for w in ResNet50_Weights:
    print(f"    - {w}")

# ---------- 其他模型对比 ----------
models_list = [
    ("EfficientNet-B0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
    ("MobileNetV3-Large", mobilenet_v3_large, MobileNet_V3_Large_Weights.DEFAULT),
    ("ViT-B/16", vit_b_16, ViT_B_16_Weights.DEFAULT),
    ("Swin-T", swin_t, Swin_T_Weights.DEFAULT),
    ("ConvNeXt-Tiny", convnext_tiny, ConvNeXt_Tiny_Weights.DEFAULT),
]

print("\n>>> 模型对比：")
for name, model_fn, w in models_list:
    m = model_fn(weights=w)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"  {name:20s}: {params:.1f}M 参数")

3.1 图像分类模型加载

>>> ResNet50
  参数量: 25.6M
  配套预处理: ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
  权重元数据: {'min_size': (1, 1), 'categories': ['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead', 'electric ray', 'stingray', 'cock', 'hen', 'ostrich', 'brambling', 'goldfinch', 'house finch', 'junco', 'indigo bunting', 'robin', 'bulbul', 'jay', 'magpie', 'chickadee', 'water ouzel', 'kite', 'bald eagle', 'vulture', 'great grey owl', 'European fire salamander', 'common newt', 'eft', 'spotted salamander', 'axolotl', 'bullfrog', 'tree frog', 'tailed frog', 'loggerhead', 'leatherback turtle', 'mud turtle', 'terrapin', 'box turtle', 'banded gecko', 'common iguana', 'American chameleon', 'whiptail', 'agama', 'frilled lizard', 'alligator lizard', 'Gila monster', 'green lizard', 'African chameleon', 'Komodo dragon', 'African crocodile', 'American alligator', '

100%|██████████| 20.5M/20.5M [00:00<00:00, 31.6MB/s]


  EfficientNet-B0     : 5.3M 参数
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:01<00:00, 15.1MB/s]


  MobileNetV3-Large   : 5.5M 参数
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:06<00:00, 53.5MB/s] 


  ViT-B/16            : 86.6M 参数
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:02<00:00, 52.1MB/s] 


  Swin-T              : 28.3M 参数
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to C:\Users\Administrator/.cache\torch\hub\checkpoints\convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:02<00:00, 56.5MB/s] 


  ConvNeXt-Tiny       : 28.6M 参数


#### 3.2 迁移学习策略

In [ ]:
# ============================================================
# 3.2 迁移学习策略
# ============================================================

import torch.nn as nn

print("\n" + "=" * 60)
print("3.2 迁移学习策略")
print("=" * 60)

# ---------- 策略1：冻结骨干 + 替换分类头 ----------
print("\n>>> 策略1：冻结骨干 + 替换分类头（最常用）")
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# 冻结所有参数
for param in model.parameters():
    param.requires_grad = False

# 替换分类头（假设 20 类）
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 20)
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"  可训练参数: {trainable:,}")
print(f"  冻结参数: {frozen:,}")

# ---------- 策略2：逐层解冻 ----------
print("\n>>> 策略2：逐层解冻（渐进式微调）")
model2 = resnet50(weights=ResNet50_Weights.DEFAULT)

# 先全部冻结
for param in model2.parameters():
    param.requires_grad = False

# 只解冻最后几层
for param in model2.layer4.parameters():
    param.requires_grad = True
model2.fc = nn.Linear(model2.fc.in_features, 20)

trainable2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"  解冻: layer4 + fc")
print(f"  可训练参数: {trainable2:,}")

# ---------- 策略3：判别式学习率 ----------
print("\n>>> 策略3：判别式学习率")
print("  backbone: lr=1e-4, 新分类头: lr=1e-3")
print("""
  optimizer = torch.optim.SGD([
      {'params': model.backbone.parameters(), 'lr': 1e-4},
      {'params': model.fc.parameters(), 'lr': 1e-3}
  ])
""")

#### 3.3 检测、分割、视频模型

In [ ]:
# ============================================================
# 3.3 检测、分割、视频模型
# ============================================================

print("\n" + "=" * 60)
print("3.3 检测、分割、视频模型")
print("=" * 60)

# ---------- Faster R-CNN (检测) ----------
print("\n>>> Faster R-CNN 目标检测")
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)

det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
det_model = fasterrcnn_resnet50_fpn(weights=det_weights)
det_model.eval()
print(f"  模型: Faster R-CNN + ResNet50-FPN")
print(f"  类别数: {len(det_weights.meta['categories'])}")

# 推理测试
demo_img = torch.rand(1, 3, 600, 800)
with torch.no_grad():
    pred = det_model(demo_img)[0]
print(f"  输出: boxes={pred['boxes'].shape}, labels={pred['labels'].shape}, scores={pred['scores'].shape}")

# ---------- DeepLabV3 (分割) ----------
print("\n>>> DeepLabV3 语义分割")
from torchvision.models.segmentation import (
    deeplabv3_resnet50,
    DeepLabV3_ResNet50_Weights
)

seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
seg_model = deeplabv3_resnet50(weights=seg_weights)
seg_model.eval()
print(f"  模型: DeepLabV3 + ResNet50")

demo_seg = torch.rand(1, 3, 520, 520)
with torch.no_grad():
    seg_out = seg_model(demo_seg)['out']
print(f"  分割输出: {seg_out.shape}  (21 类)")

# ---------- R3D-18 (视频) ----------
print("\n>>> R3D-18 视频分类")
from torchvision.models.video import r3d_18, R3D_18_Weights

video_weights = R3D_18_Weights.DEFAULT
video_model = r3d_18(weights=video_weights)
video_model.eval()
print(f"  模型: 3D ResNet-18")
print(f"  输入格式: (B, C, T, H, W) = (batch, 3, 16, 112, 112)")

#### 3.4 模型导出与量化

In [ ]:
# ============================================================
# 3.4 模型导出与量化
# ============================================================

print("\n" + "=" * 60)
print("3.4 模型导出与量化")
print("=" * 60)

# ---------- TorchScript 导出 ----------
print("\n>>> TorchScript 导出")
model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.eval()

example_input = torch.rand(1, 3, 224, 224)

# Trace 模式
traced_model = torch.jit.trace(model, example_input)
print(f"  Trace 导出: {type(traced_model).__name__}")

# Script 模式（支持控制流）
scripted_model = torch.jit.script(model)
print(f"  Script 导出: {type(scripted_model).__name__}")

print("  保存: traced_model.save('model.pt')")
print("  加载: torch.jit.load('model.pt')")

# ---------- 量化模型 ----------
print("\n>>> 量化模型")
from torchvision.models.quantization import (
    mobilenet_v3_large as quant_mobilenet_v3_large,
    MobileNet_V3_Large_QuantizedWeights
)

quant_weights = MobileNet_V3_Large_QuantizedWeights.DEFAULT
quant_model = quant_mobilenet_v3_large(weights=quant_weights, quantize=True)
print(f"  MobileNetV3-Large (Quantized)")
print(f"  优势: 模型大小减小约75%，推理速度提升2-3倍")

# ---------- 模型选择建议 ----------
print("\n>>> 模型选择建议")
print("  精度优先: ConvNeXt, Swin Transformer, EfficientNetV2")
print("  速度优先: MobileNetV3, ShuffleNetV2")
print("  平衡方案: ResNet50, EfficientNet-B3")
print("  检测任务: Faster R-CNN (精度) / SSD (速度)")
print("  分割任务: DeepLabV3 (通用) / LRASPP (实时)")

#### 第四章：torchvision.utils — 可视化工具

In [ ]:
# ============================================================
# 4.1 make_grid 和 save_image
# ============================================================

from torchvision import utils

print("=" * 60)
print("4.1 make_grid 和 save_image")
print("=" * 60)

# ---------- make_grid ----------
batch = torch.rand(16, 3, 64, 64)  # 16 张 64x64 图片

grid = utils.make_grid(
    batch,
    nrow=4,              # 每行 4 张
    padding=2,           # 间距 2 像素
    normalize=True,      # 归一化到 [0, 1]
    pad_value=1.0        # 填充白色
)

print(f"  输入: {batch.shape}")
print(f"  输出网格: {grid.shape}  (RGB 图片)")
print(f"  网格尺寸: {grid.shape[1]}x{grid.shape[2]} 像素")

# ---------- save_image ----------
single_img = torch.rand(3, 256, 256)
# utils.save_image(single_img, 'output.png')
print(f"\n  保存图片: save_image(tensor, 'filename.png')")
print(f"  支持格式: PNG, JPEG, BMP")

In [ ]:
# ============================================================
# 4.2 检测框、分割掩码、关键点可视化
# ============================================================

from torchvision.utils import (
    draw_bounding_boxes,
    draw_segmentation_masks,
    draw_keypoints
)

print("\n" + "=" * 60)
print("4.2 检测框、分割掩码、关键点可视化")
print("=" * 60)

# 创建测试图片 (uint8, [0, 255])
img = (torch.rand(3, 400, 600) * 255).to(torch.uint8)
print(f"测试图片: {img.shape}, dtype={img.dtype}")

# ---------- 绘制边界框 ----------
boxes = torch.tensor([
    [50, 50, 200, 250],
    [250, 100, 450, 300],
    [100, 250, 300, 380]
])
labels = ["cat", "dog", "bird"]
colors = ["red", "blue", "green"]

img_with_boxes = draw_bounding_boxes(
    img, boxes, labels=labels, colors=colors, width=3
)
print(f"  绘制框后: {img_with_boxes.shape}")

# ---------- 绘制分割掩码 ----------
masks = torch.zeros(3, 400, 600, dtype=torch.bool)
masks[0, 30:150, 30:200] = True
masks[1, 150:300, 200:400] = True
masks[2, 250:380, 350:550] = True

img_with_masks = draw_segmentation_masks(
    img, masks, alpha=0.5, colors=["red", "green", "blue"]
)
print(f"  绘制掩码后: {img_with_masks.shape}")

# ---------- 绘制关键点 ----------
keypoints = torch.tensor([
    [300, 50, 2], [280, 80, 2], [320, 80, 2],
    [260, 100, 2], [340, 100, 2], [280, 150, 2],
    [320, 150, 2], [260, 220, 2], [340, 220, 2],
    [270, 290, 2], [330, 290, 2], [280, 300, 2],
    [320, 300, 2], [280, 350, 1], [320, 350, 2],
    [270, 390, 2], [330, 390, 2]
])

connectivity = [(0,1), (0,2), (1,3), (2,4), (5,6), (5,7), (7,9), (6,8), (8,10), (5,11), (6,12), (11,12), (11,13), (13,15), (12,14), (14,16)]

img_with_kps = draw_keypoints(
    img, keypoints.unsqueeze(0), connectivity=connectivity,
    colors="yellow", radius=4, width=3
)
print(f"  绘制关键点后: {img_with_kps.shape}")
print(f"  关键点数量: {len(keypoints)}")
print(f"  visibility: 2=可见, 1=遮挡, 0=不可见")

#### 第五章：torchvision.ops — 底层视觉算子

In [ ]:
# ============================================================
# 5.1 NMS 与边界框操作
# ============================================================

from torchvision.ops import (
    nms, batched_nms,
    box_iou, generalized_box_iou, complete_box_iou,
    box_convert, box_area,
    clip_boxes_to_image, remove_small_boxes
)

print("=" * 60)
print("5.1 NMS 与边界框操作")
print("=" * 60)

# ---------- NMS 非极大值抑制 ----------
boxes = torch.tensor([
    [10, 10, 100, 100],
    [15, 15, 95, 95],
    [80, 80, 160, 160],
    [200, 200, 300, 300],
    [210, 210, 290, 290]
], dtype=torch.float32)
scores = torch.tensor([0.95, 0.85, 0.75, 0.9, 0.8])

keep = nms(boxes, scores, iou_threshold=0.5)
print(f"\n>>> NMS")
print(f"  总框数: {len(boxes)}")
print(f"  保留: {len(keep)} 个")
print(f"  保留索引: {keep.tolist()}")

# 不同阈值对比
for thresh in [0.3, 0.5, 0.7]:
    k = nms(boxes, scores, iou_threshold=thresh)
    print(f"  IoU={thresh}: 保留 {len(k)} 个")

# ---------- IoU 计算 ----------
print("\n>>> IoU 计算")
pred_boxes = torch.tensor([[20, 30, 100, 150], [50, 50, 120, 130]], dtype=torch.float32)
gt_boxes = torch.tensor([[25, 35, 95, 145], [200, 200, 300, 300]], dtype=torch.float32)

iou = box_iou(pred_boxes, gt_boxes)
print(f"  IoU 矩阵:\n{iou}")

# ---------- 框格式转换 ----------
print("\n>>> 框格式转换")
boxes_xyxy = torch.tensor([[10, 20, 100, 150]], dtype=torch.float32)
boxes_xywh = box_convert(boxes_xyxy, in_fmt='xyxy', out_fmt='xywh')
boxes_cxcywh = box_convert(boxes_xyxy, in_fmt='xyxy', out_fmt='cxcywh')

print(f"  XYXY:  {boxes_xyxy[0].tolist()}")
print(f"  XYWH:  {boxes_xywh[0].tolist()}")
print(f"  CXCYWH: {boxes_cxcywh[0].tolist()}")

In [ ]:
# ============================================================
# 5.2 ROI 池化与损失函数
# ============================================================

from torchvision.ops import roi_align, roi_pool, sigmoid_focal_loss, giou_loss

print("\n" + "=" * 60)
print("5.2 ROI 池化与损失函数")
print("=" * 60)

# ---------- ROI Align ----------
feature_map = torch.rand(1, 256, 50, 50)
rois = torch.tensor([
    [0, 10, 15, 30, 40],
    [0, 5, 5, 25, 25]
], dtype=torch.float32)

pooled = roi_align(feature_map, rois, output_size=(7, 7), spatial_scale=1.0)
print(f"\n>>> ROI Align")
print(f"  特征图: {feature_map.shape}")
print(f"  ROI 数量: {len(rois)}")
print(f"  输出: {pooled.shape}")

# ---------- Focal Loss ----------
print("\n>>> Focal Loss")
logits = torch.tensor([[2.0, -1.0, 0.5], [-0.5, 1.5, -2.0], [0.1, 0.2, 3.0]])
targets = torch.tensor([0, 1, 2])

loss = sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0, reduction='mean')
print(f"  Focal Loss: {loss.item():.4f}")
print(f"  参数: alpha=0.25 (平衡权重), gamma=2.0 (聚焦参数)")

# ---------- GIoU Loss ----------
print("\n>>> GIoU Loss")
pred = torch.tensor([[20, 30, 100, 150]], dtype=torch.float32)
target = torch.tensor([[25, 35, 95, 145]], dtype=torch.float32)

giou = giou_loss(pred, target)
print(f"  GIoU Loss: {giou.item():.4f}")
print(f"  预测框: {pred[0].tolist()}")
print(f"  目标框: {target[0].tolist()}")

#### 第六章：torchvision.io — 张量式 IO 读写

In [ ]:
# ============================================================
# 6.1 图片 IO 操作
# ============================================================

from torchvision import io

print("=" * 60)
print("6.1 图片 IO 操作")
print("=" * 60)

# 创建测试图片
with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
    test_img = Image.fromarray(
        np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
    )
    test_img.save(tmp.name)
    tmp_path = tmp.name

try:
    # ---------- read_image ----------
    print("\n>>> read_image 读取图片")
    img_tensor = io.read_image(tmp_path)
    print(f"  文件: {tmp_path}")
    print(f"  张量形状: {img_tensor.shape}")
    print(f"  数据类型: {img_tensor.dtype}")
    print(f"  值范围: [{img_tensor.min()}, {img_tensor.max()}]")

    # ---------- 编码与解码 ----------
    print("\n>>> JPEG 编码与解码")
    jpeg_bytes = io.encode_jpeg(img_tensor, quality=75)
    print(f"  JPEG 编码后: {len(jpeg_bytes):,} bytes")

    decoded = io.decode_jpeg(jpeg_bytes)
    print(f"  解码后形状: {decoded.shape}")

    # 不同质量对比
    print("\n  不同质量参数对比：")
    for q in [10, 50, 90]:
        size = len(io.encode_jpeg(img_tensor, quality=q))
        print(f"    quality={q}: {size:,} bytes")

    # ---------- PNG 编码 ----------
    print("\n>>> PNG 编码")
    png_bytes = io.encode_png(img_tensor)
    print(f"  PNG 编码后: {len(png_bytes):,} bytes")
    print(f"  PNG 无损压缩，文件通常比 JPEG 大")

    # ---------- 写入文件 ----------
    print("\n>>> 写入文件")
    out_jpg = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False)
    io.write_jpeg(img_tensor, out_jpg.name, quality=90)
    print(f"  保存 JPEG: {out_jpg.name}")
    print(f"  文件大小: {os.path.getsize(out_jpg.name):,} bytes")
    os.unlink(out_jpg.name)

finally:
    os.unlink(tmp_path)

print("\n✅ 图片 IO 测试完成")

In [ ]:
# ============================================================
# 6.2 视频 IO 操作（需要 ffmpeg）
# ============================================================

print("\n" + "=" * 60)
print("6.2 视频 IO 操作")
print("=" * 60)

print("""
📹 视频 IO 函数：

  read_video(path, start_pts=0, end_pts=None, pts_unit='sec')
    返回: (video, audio, info)
    video: Tensor (T, H, W, C) uint8
    audio: Tensor (S, C) 或 None
    info: Dict {video_fps, audio_fps}

  read_video_timestamps(path)
    返回: (pts, video_fps)  # 用于获取帧数和帧率

  write_video(path, frames, fps, video_codec='libx264')
    写入视频文件
""")

# 模拟视频帧处理
print("\n>>> 模拟视频帧处理")
num_frames, h, w = 30, 240, 320
mock_video = torch.randint(0, 256, (num_frames, h, w, 3), dtype=torch.uint8)

print(f"  模拟视频: {num_frames} 帧, {h}x{w}")
print(f"  张量形状: {mock_video.shape}")

# 帧采样策略
print("\n>>> 帧采样策略")
stride = 3
sampled = mock_video[::stride]
print(f"  均匀采样 (stride={stride}): {len(mock_video)} → {len(sampled)} 帧")

random_idx = torch.randperm(num_frames)[:8].sort()[0]
random_sampled = mock_video[random_idx]
print(f"  随机采样 (8帧): 索引={random_idx.tolist()}")

print("\n📌 注意事项：")
print("  1. 视频 IO 需要系统安装 ffmpeg")
print("  2. Linux: sudo apt install ffmpeg")
print("  3. macOS: brew install ffmpeg")
print("  4. Windows: conda install ffmpeg")

#### 第七章：tv_tensors — v2 专用张量类型

In [ ]:
# ============================================================
# 7. tv_tensors 基础用法
# ============================================================

from torchvision import tv_tensors

print("=" * 60)
print("7. tv_tensors 张量类型")
print("=" * 60)

# ---------- tv_tensors.Image ----------
print("\n>>> tv_tensors.Image")
img_data = torch.randint(0, 256, (3, 224, 224), dtype=torch.uint8)
tv_img = tv_tensors.Image(img_data)
print(f"  类型: {type(tv_img).__name__}")
print(f"  形状: {tv_img.shape}")
print(f"  dtype: {tv_img.dtype}")

# ---------- tv_tensors.BoundingBoxes ----------
print("\n>>> tv_tensors.BoundingBoxes")
boxes_data = torch.tensor([
    [10, 20, 100, 200],
    [50, 60, 150, 180]
], dtype=torch.float32)

tv_boxes = tv_tensors.BoundingBoxes(
    boxes_data,
    format="XYXY",
    canvas_size=(480, 640)
)

print(f"  类型: {type(tv_boxes).__name__}")
print(f"  形状: {tv_boxes.shape}")
print(f"  格式: {tv_boxes.format}")
print(f"  画布大小: {tv_boxes.canvas_size}")

# 支持的格式
print("\n  支持的框格式:")
print("    XYXY:  (x1, y1, x2, y2) - 左上右下")
print("    XYWH:  (x, y, w, h) - 左上+宽高")
print("    CXCYWH: (cx, cy, w, h) - 中心+宽高")

# ---------- tv_tensors.Mask ----------
print("\n>>> tv_tensors.Mask")
mask_data = torch.randint(0, 2, (3, 480, 640), dtype=torch.uint8)
tv_mask = tv_tensors.Mask(mask_data)
print(f"  类型: {type(tv_mask).__name__}")
print(f"  形状: {tv_mask.shape}")
print(f"  唯一值: {tv_mask.unique().tolist()}")

# ---------- tv_tensors.Video ----------
print("\n>>> tv_tensors.Video")
video_data = torch.randint(0, 256, (16, 3, 112, 112), dtype=torch.uint8)
tv_video = tv_tensors.Video(video_data)
print(f"  类型: {type(tv_video).__name__}")
print(f"  形状: {tv_video.shape}")

print("\n✨ tv_tensors 核心优势：")
print("  1. 保留元数据（格式、画布尺寸等）")
print("  2. v2 变换自动识别并同步变换")
print("  3. 继承自 torch.Tensor，兼容所有操作")

#### 第八章：完整训练流程示例

In [ ]:
# ============================================================
# 8.1 完整训练流程（分类任务）
# 使用 CIFAR10 + ResNet18
# ============================================================

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

print("=" * 60)
print("8.1 完整训练流程")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

# ---------- 超参数 ----------
BATCH_SIZE = 64
NUM_EPOCHS = 3  # 演示用，实际训练需要更多 epoch
LEARNING_RATE = 1e-3
NUM_CLASSES = 10

# ---------- 数据准备 ----------
print("\n>>> 数据准备")

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])

full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
train_set, val_set = random_split(full_train, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"  训练集: {len(train_set)} 样本")
print(f"  验证集: {len(val_set)} 样本")
print(f"  测试集: {len(test_set)} 样本")

# ---------- 模型 ----------
print("\n>>> 模型构建")
from torchvision.models import resnet18

model = resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

print(f"  模型: ResNet18 (适配 CIFAR10)")
print(f"  参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# ---------- 训练 ----------
print("\n>>> 开始训练")
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += targets.size(0)
        correct += pred.eq(targets).sum().item()
    return total_loss / len(loader), 100 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += targets.size(0)
            correct += pred.eq(targets).sum().item()
    return total_loss / len(loader), 100 * correct / total

print("-" * 50)
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f}, Acc: {val_acc:.2f}%")
print("-" * 50)

# ---------- 测试 ----------
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"\n测试准确率: {test_acc:.2f}%")

In [ ]:
# ============================================================
# 8.2 推理结果可视化
# ============================================================

print("\n" + "=" * 60)
print("8.2 推理结果可视化")
print("=" * 60)

model.eval()

# 取一个 batch 进行推理
data_iter = iter(test_loader)
images, labels = next(data_iter)
images, labels = images[:16], labels[:16]

with torch.no_grad():
    outputs = model(images.to(device))
    _, predictions = outputs.max(1)

# 反归一化用于可视化
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
vis_images = torch.clamp(images * std + mean, 0, 1)

# 制作网格
grid = utils.make_grid(vis_images, nrow=4, padding=2)

# 保存
utils.save_image(grid, "predictions.jpg")
print(f"  已保存预测结果到: predictions.jpg")

classes = test_set.dataset.classes if hasattr(test_set, 'dataset') else test_set.classes
print("\n  预测结果：")
for i in range(len(images)):
    correct = "✓" if labels[i] == predictions[i] else "✗"
    print(f"    [{correct}] 真实: {classes[labels[i]]:10s} → 预测: {classes[predictions[i].item()]:10s}")

acc = 100 * (predictions.cpu() == labels).sum().item() / len(labels)
print(f"\n  本 batch 准确率: {acc:.1f}%")

#### 第九章：总结与最佳实践

In [ ]:
# ============================================================
# 9. 总结与最佳实践
# ============================================================

print("=" * 70)
print("TorchVision 总结与最佳实践")
print("=" * 70)

print("""
📚 学习路线图：

  入门阶段
  ├── datasets: 加载标准数据集
  ├── transforms: 基础图像预处理
  └── models: 加载预训练模型

  进阶阶段
  ├── transforms.v2: 检测/分割同步增强
  ├── tv_tensors: 带元数据的张量类型
  ├── utils: 结果可视化
  └── 迁移学习: 微调预训练模型

  高级阶段
  ├── ops: 自定义检测器算子
  ├── io: 高性能数据读取
  ├── 模型导出: TorchScript/量化
  └── 完整训练流程: 从数据到部署
""")

print("\n✅ 核心要点：")
print("  1. TorchVision 覆盖 CV 全链路：数据→增强→模型→训练→推理→可视化")
print("  2. v2 是未来：新项目应优先使用 transforms.v2 + tv_tensors")
print("  3. 权重系统：使用 Weights.DEFAULT 和 weights.transforms()")
print("  4. 检测/分割：必须使用 v2 的同步变换机制")
print("  5. 性能优先：io 模块直接张量读写，ops 模块 CUDA 加速")
print("  6. 官方文档：https://pytorch.org/vision/stable/")

print("\n" + "=" * 70)
print("🎉 教程完成！所有代码均可独立执行。")
print("=" * 70)